# Graded Assignment 3: 9 to 5

Time to show off your SQL skills! For each question, copy the SQL query you used and make note of the answer.

## The Dataset

For this assignment, you will be using the Bureau of Labor Statistics (BLS) Current Employment Survey (CES) results which can be found on [Kaggle](https://www.kaggle.com/datasets/bls/employment).

## Business Issue

You are working for the Bureau of Labor Statistics with the United States government and have been approached by your boss with an important meeting request. You have been asked by your supervisor to meet with Dolly Parton whose nonprofit is looking to shed light on the state of employment in the United States. As part of the 9 to 5 project, their research is focused on production and nonsupervisory employees and how those employees fare compared to all employees in the United States. While the data the BLS collects from the CES is publicly available, Dolly Parton and her colleagues need your assistance navigating the thousands of rows in each table in LaborStatisticsDB.

## About the Dataset

This dataset comes directly from the Bureau of Labor Statistics’ Current Employment Survey (CES). Here are some things you need to know:

1. The industry table contains an NAICS code. This is different from the industry code. NAICS stands for North American Industry Classification System.
1. Series ID is composed of multiple different codes. CES stands for Current Employment Survey, the name of the survey which collected the data. The industry code as specified by the BLS and the data type code as specified in the datatype table.

## Set Up

To connect to the database, use the same connection info used during the SQL lessons. 

For the assignment, we will be using `LaborStatisticsDB`.

## Database Exploration

To start with, let’s get to know the database further.

1. Use this space to make note of each table in the database, the columns within each table, each column’s data type, and how the tables are connected. You can write this down or draw a diagram. Whatever method helps you get an understanding of what is going on with `LaborStatisticsDB`.
   
   To add a photo, diagram or document to your file, drop the file into the folder that holds this notebook.  Use the link button to the right of the  </> symbol in the gray part of this cell, the link is just the name of your file.

In [ ]:
--Screenshot is the file 95-database-key.png in this subdirectory

2. What is the datatype for women employees?

In [ ]:
--10

SELECT *
FROM LaborStatisticsDB.dbo.datatype
WHERE data_type_text LIKE 'women employees'

3. What is the series id for  women employees in the commercial banking industry in the financial activities supersector?

In [ ]:
-- CES5552211010

SELECT *
FROM LaborStatisticsDB.dbo.series
WHERE industry_code IN 
        (
        SELECT industry_code
        FROM LaborStatisticsDB.dbo.industry
        WHERE industry_name LIKE 'Commercial banking'
        )
    AND supersector_code IN 
        (
        SELECT supersector_code
        FROM LaborStatisticsDB.dbo.supersector
        WHERE supersector_name LIKE 'financial activities'
        )
    AND data_type_code IN 
        (
        SELECT data_type_code
        FROM LaborStatisticsDB.dbo.datatype
        WHERE data_type_text LIKE 'women employees'
        )

## Aggregate Your Friends and Code some SQL

Put together the following:

1. How many employees were reported in 2016 in all industries? Round to the nearest whole number.

In [ ]:
-- 2,340,612 

SELECT ROUND(SUM (value), 0) AS total_employees
FROM LaborStatisticsDB.dbo.annual_2016
WHERE series_ID IN (
        SELECT series_ID
        FROM LaborStatisticsDB.dbo.series
        WHERE data_type_code = '1'
)

2. How many women employees were reported in 2016 in all industries? Round to the nearest whole number. 

In [ ]:
-- 1,125,490

SELECT ROUND(SUM (value), 0) AS total_women_employees
FROM LaborStatisticsDB.dbo.annual_2016
WHERE series_ID IN (
        SELECT series_ID
        FROM LaborStatisticsDB.dbo.series
        WHERE data_type_code = '10'
)

3. How many production/nonsupervisory employees were reported in 2016? Round to the nearest whole number. 

In [ ]:
-- 1,263,650

SELECT ROUND(SUM (value), 0) AS total_production_employees
FROM LaborStatisticsDB.dbo.annual_2016
WHERE series_ID IN (
        SELECT series_ID
        FROM LaborStatisticsDB.dbo.series
        WHERE data_type_code = '6'
)

4. In January 2017, what is the average weekly hours worked by production and nonsupervisory employees across all industries?

In [ ]:
-- 36.1

SELECT ROUND(AVG(value),1) AS avg_hours_worked
FROM LaborStatisticsDB.dbo.january_2017
WHERE series_id IN 
        (
        SELECT series_id
        FROM LaborStatisticsDB.dbo.series
        WHERE data_type_code = '7'
        )

5. What is the total weekly payroll for production and nonsupervisory employees across all industries in January 2017? Round to the nearest penny.

In [ ]:
-- $1,838,753,220 ; no rounding occured

SELECT ROUND(SUM(value),2) AS total_weekly_payroll
FROM LaborStatisticsDB.dbo.january_2017
WHERE series_id IN 
        (
        SELECT series_id
        FROM LaborStatisticsDB.dbo.series
        WHERE data_type_code = '82'
        )


6. In January 2017, for which industry was the average weekly hours worked by production and nonsupervisory employees the highest? Which industry was the lowest?

In [ ]:
-- Highest: Motor vehicle power train components

SELECT TOP 1 * 
FROM LaborStatisticsDB.dbo.january_2017
WHERE series_id IN
    (
        SELECT series_id
        FROM LaborStatisticsDB.dbo.series
        WHERE data_type_code = '7'
    )
ORDER BY "value" DESC ;

-- returns series_id CES3133635007

SELECT industry_name
FROM LaborStatisticsDB.dbo.industry
WHERE industry_code IN
    (        SELECT industry_code
        FROM LaborStatisticsDB.dbo.series
        WHERE series_id IN 
        (
            SELECT series_id
            FROM LaborStatisticsDB.dbo.january_2017
            WHERE series_id = 'CES3133635007'

        )
    )

-- Lowest: Fitness and recreational sports centers

SELECT TOP 1 * 
FROM LaborStatisticsDB.dbo.january_2017
WHERE series_id IN
    (
        SELECT series_id
        FROM LaborStatisticsDB.dbo.series
        WHERE data_type_code = '7'
    )
ORDER BY "value"

-- returns series_id CEU7071394007

SELECT industry_name
FROM LaborStatisticsDB.dbo.industry
WHERE industry_code IN
    (        SELECT industry_code
        FROM LaborStatisticsDB.dbo.series
        WHERE series_id IN 
        (
            SELECT series_id
            FROM LaborStatisticsDB.dbo.january_2017
            WHERE series_id = 'CEU7071394007'

        )
    )



7. In January 2021, for which industry was the total weekly payroll for production and nonsupervisory employees the highest? Which industry was the lowest?

In [ ]:
-- going off the assumption that this should also be January 2017
-- Highest: Professional and business services

SELECT TOP 1 *
FROM LaborStatisticsDB.dbo.january_2017
WHERE series_id IN
    (
        SELECT series_id
        FROM LaborStatisticsDB.dbo.series
        WHERE data_type_code = '82'
    )
   AND original_file NOT LIKE '%all%'
   AND original_file NOT LIKE '%total%'
ORDER BY "value" DESC

-- returns series_id CES6000000082

SELECT industry_name
FROM LaborStatisticsDB.dbo.industry
WHERE industry_code IN
    (        SELECT industry_code
        FROM LaborStatisticsDB.dbo.series
        WHERE series_id IN 
        (
            SELECT series_id
            FROM LaborStatisticsDB.dbo.january_2017
            WHERE series_id = 'CES6000000082'

        )
    )

-- Lowest: Coin-operated laundries and drycleaners

SELECT TOP 1 *
FROM LaborStatisticsDB.dbo.january_2017
WHERE series_id IN
    (
        SELECT series_id
        FROM LaborStatisticsDB.dbo.series
        WHERE data_type_code = '82'
    )
   AND original_file NOT LIKE '%all%'
   AND original_file NOT LIKE '%total%'
ORDER BY "value"

-- returns series_id CEU8081231082

SELECT industry_name
FROM LaborStatisticsDB.dbo.industry
WHERE industry_code IN
    (        SELECT industry_code
        FROM LaborStatisticsDB.dbo.series
        WHERE series_id IN 
        (
            SELECT series_id
            FROM LaborStatisticsDB.dbo.january_2017
            WHERE series_id = 'CEU8081231082'

        )
    )


## Join in on the Fun

Time to start joining! You can choose the type of join you use, just make sure to make a  note!

1. Join `annual_2016` with `series` on `series_id`. We only want the data in the `annual_2016` table to be included in the result.

In [ ]:
--Displaying top 50 results

SELECT *
FROM LaborStatisticsDB.dbo.annual_2016 AS a
LEFT JOIN LaborStatisticsDB.dbo.series AS s
ON s.series_id = a.series_id
ORDER BY a.id

/*
0	CEU5500000007	2016	M13	36.9	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5500000007	55	55000000	7	U	Average weekly hours of production and nonsupervisory employees
1	CEU5500000008	2016	M13	26.11	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5500000008	55	55000000	8	U	Average hourly earnings of production and nonsupervisory employees
2	CEU5500000030	2016	M13	962.73	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5500000030	55	55000000	30	U	Average weekly earnings of production and nonsupervisory employees
3	CEU5500000031	2016	M13	411.29	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5500000031	55	55000000	31	U	Average weekly earnings of production and nonsupervisory employees
4	CEU5500000032	2016	M13	11.15	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5500000032	55	55000000	32	U	Average hourly earnings of production and nonsupervisory employees
5	CEU5500000034	2016	M13	111.6	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5500000034	55	55000000	34	U	Indexes of aggregate weekly hours of production and nonsupervisory employees
6	CEU5500000035	2016	M13	179.2	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5500000035	55	55000000	35	U	Indexes of aggregate weekly payrolls of production and nonsupervisory employees
7	CEU5500000081	2016	M13	236997	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5500000081	55	55000000	81	U	Aggregate weekly hours of production and nonsupervisory employees
8	CEU5500000082	2016	M13	6189003	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5500000082	55	55000000	82	U	Aggregate weekly payrolls of production and nonsupervisory employees
9	CEU5552200007	2016	M13	37.3	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552200007	55	55522000	7	U	Average weekly hours of production and nonsupervisory employees
10	CEU5552200008	2016	M13	22.37	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552200008	55	55522000	8	U	Average hourly earnings of production and nonsupervisory employees
11	CEU5552200030	2016	M13	833.43	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552200030	55	55522000	30	U	Average weekly earnings of production and nonsupervisory employees
12	CEU5552200031	2016	M13	356.05	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552200031	55	55522000	31	U	Average weekly earnings of production and nonsupervisory employees
13	CEU5552200032	2016	M13	9.56	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552200032	55	55522000	32	U	Average hourly earnings of production and nonsupervisory employees
14	CEU5552200034	2016	M13	104.2	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552200034	55	55522000	34	U	Indexes of aggregate weekly hours of production and nonsupervisory employees
15	CEU5552200035	2016	M13	162.5	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552200035	55	55522000	35	U	Indexes of aggregate weekly payrolls of production and nonsupervisory employees
16	CEU5552200081	2016	M13	73459	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552200081	55	55522000	81	U	Aggregate weekly hours of production and nonsupervisory employees
17	CEU5552200082	2016	M13	1643184	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552200082	55	55522000	82	U	Aggregate weekly payrolls of production and nonsupervisory employees
18	CEU5552210007	2016	M13	37	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552210007	55	55522100	7	U	Average weekly hours of production and nonsupervisory employees
19	CEU5552210008	2016	M13	20.25	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552210008	55	55522100	8	U	Average hourly earnings of production and nonsupervisory employees
20	CEU5552210030	2016	M13	749.49	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552210030	55	55522100	30	U	Average weekly earnings of production and nonsupervisory employees
21	CEU5552210031	2016	M13	320.19	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552210031	55	55522100	31	U	Average weekly earnings of production and nonsupervisory employees
22	CEU5552210032	2016	M13	8.65	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552210032	55	55522100	32	U	Average hourly earnings of production and nonsupervisory employees
23	CEU5552210034	2016	M13	101.8	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552210034	55	55522100	34	U	Indexes of aggregate weekly hours of production and nonsupervisory employees
24	CEU5552210035	2016	M13	161.7	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552210035	55	55522100	35	U	Indexes of aggregate weekly payrolls of production and nonsupervisory employees
25	CEU5552210081	2016	M13	46102	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552210081	55	55522100	81	U	Aggregate weekly hours of production and nonsupervisory employees
26	CEU5552210082	2016	M13	933434	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552210082	55	55522100	82	U	Aggregate weekly payrolls of production and nonsupervisory employees
27	CEU5552211007	2016	M13	37.1	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552211007	55	55522110	7	U	Average weekly hours of production and nonsupervisory employees
28	CEU5552211008	2016	M13	20.35	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552211008	55	55522110	8	U	Average hourly earnings of production and nonsupervisory employees
29	CEU5552211030	2016	M13	754.09	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552211030	55	55522110	30	U	Average weekly earnings of production and nonsupervisory employees
30	CEU5552211031	2016	M13	322.16	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552211031	55	55522110	31	U	Average weekly earnings of production and nonsupervisory employees
31	CEU5552211032	2016	M13	8.69	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552211032	55	55522110	32	U	Average hourly earnings of production and nonsupervisory employees
32	CEU5552211034	2016	M13	106.8	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552211034	55	55522110	34	U	Indexes of aggregate weekly hours of production and nonsupervisory employees
33	CEU5552211035	2016	M13	173.2	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552211035	55	55522110	35	U	Indexes of aggregate weekly payrolls of production and nonsupervisory employees
34	CEU5552211081	2016	M13	35082	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552211081	55	55522110	81	U	Aggregate weekly hours of production and nonsupervisory employees
35	CEU5552211082	2016	M13	713870	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552211082	55	55522110	82	U	Aggregate weekly payrolls of production and nonsupervisory employees
36	CEU5552212007	2016	M13	36.4	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552212007	55	55522120	7	U	Average weekly hours of production and nonsupervisory employees
37	CEU5552212008	2016	M13	20.53	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552212008	55	55522120	8	U	Average hourly earnings of production and nonsupervisory employees
38	CEU5552212030	2016	M13	746.56	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552212030	55	55522120	30	U	Average weekly earnings of production and nonsupervisory employees
39	CEU5552212031	2016	M13	318.94	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552212031	55	55522120	31	U	Average weekly earnings of production and nonsupervisory employees
40	CEU5552212032	2016	M13	8.77	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552212032	55	55522120	32	U	Average hourly earnings of production and nonsupervisory employees
41	CEU5552212034	2016	M13	52.4	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552212034	55	55522120	34	U	Indexes of aggregate weekly hours of production and nonsupervisory employees
42	CEU5552212035	2016	M13	79	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552212035	55	55522120	35	U	Indexes of aggregate weekly payrolls of production and nonsupervisory employees
43	CEU5552212081	2016	M13	3299	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552212081	55	55522120	81	U	Aggregate weekly hours of production and nonsupervisory employees
44	CEU5552212082	2016	M13	67744	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552212082	55	55522120	82	U	Aggregate weekly payrolls of production and nonsupervisory employees
45	CEU5552219007	2016	M13	37.1	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552219007	55	55522190	7	U	Average weekly hours of production and nonsupervisory employees
46	CEU5552219008	2016	M13	19.66	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552219008	55	55522190	8	U	Average hourly earnings of production and nonsupervisory employees
47	CEU5552219030	2016	M13	730.09	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552219030	55	55522190	30	U	Average weekly earnings of production and nonsupervisory employees
48	CEU5552219031	2016	M13	311.9	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552219031	55	55522190	31	U	Average weekly earnings of production and nonsupervisory employees
49	CEU5552219032	2016	M13	8.4	NULL	ce.data.55c.FinancialActivities.ProductionEmployeeHoursAndEarnings.csv	CEU5552219032	55	55522190	32	U	Average hourly earnings of production and nonsupervisory employees
*/

Commands completed successfully.

Total execution time: 00:00:00.019

2. Join `series` and `datatype` on `data_type_code`.

In [ ]:
--Displaying top 50 results

SELECT T*
FROM LaborStatisticsDB.dbo.series AS s
LEFT JOIN LaborStatisticsDB.dbo.datatype AS d
ON d.data_type_code = s.data_type_code
ORDER BY s.series_id

/*
CES0000000001	0	00000000	1	S	All employees	1	ALL EMPLOYEES
CES0000000010	0	00000000	10	S	Women employees	10	WOMEN EMPLOYEES
CES0000000025	0	00000000	25	S	All employees	25	ALL EMPLOYEES
CES0000000026	0	00000000	26	S	All employees	26	ALL EMPLOYEES
CES0500000001	5	05000000	1	S	All employees	1	ALL EMPLOYEES
CES0500000002	5	05000000	2	S	Average weekly hours of all employees	2	AVERAGE WEEKLY HOURS OF ALL EMPLOYEES
CES0500000003	5	05000000	3	S	Average hourly earnings of all employees	3	AVERAGE HOURLY EARNINGS OF ALL EMPLOYEES
CES0500000006	5	05000000	6	S	Production and nonsupervisory employees	6	PRODUCTION AND NONSUPERVISORY EMPLOYEES
CES0500000007	5	05000000	7	S	Average weekly hours of production and nonsupervisory employees	7	AVERAGE WEEKLY HOURS OF PRODUCTION AND NONSUPERVISORY EMPLOYEES
CES0500000008	5	05000000	8	S	Average hourly earnings of production and nonsupervisory employees	8	AVERAGE HOURLY EARNINGS OF PRODUCTION AND NONSUPERVISORY EMPLOYEES
CES0500000010	5	05000000	10	S	Women employees	10	WOMEN EMPLOYEES
CES0500000011	5	05000000	11	S	Average weekly earnings of all employees	11	AVERAGE WEEKLY EARNINGS OF ALL EMPLOYEES
CES0500000012	5	05000000	12	S	Average weekly earnings of all employees	12	AVERAGE WEEKLY EARNINGS OF ALL EMPLOYEES
CES0500000013	5	05000000	13	S	Average hourly earnings of all employees	13	AVERAGE HOURLY EARNINGS OF ALL EMPLOYEES
CES0500000016	5	05000000	16	S	Indexes of aggregate weekly hours of all employees	16	INDEXES OF AGGREGATE WEEKLY HOURS OF ALL EMPLOYEES
CES0500000017	5	05000000	17	S	Indexes of aggregate weekly payrolls of all employees	17	INDEXES OF AGGREGATE WEEKLY PAYROLLS OF ALL EMPLOYEES
CES0500000019	5	05000000	19	S	Average weekly hours of all employees	19	AVERAGE WEEKLY HOURS OF ALL EMPLOYEES
CES0500000021	5	05000000	21	S	Diffusion indexes	21	DIFFUSION INDEXES
CES0500000022	5	05000000	22	S	Diffusion indexes	22	DIFFUSION INDEXES
CES0500000023	5	05000000	23	S	Diffusion indexes	23	DIFFUSION INDEXES
CES0500000025	5	05000000	25	S	All employees	25	ALL EMPLOYEES
CES0500000026	5	05000000	26	S	All employees	26	ALL EMPLOYEES
CES0500000030	5	05000000	30	S	Average weekly earnings of production and nonsupervisory employees	30	AVERAGE WEEKLY EARNINGS OF PRODUCTION AND NONSUPERVISORY EMPLOYEES
CES0500000031	5	05000000	31	S	Average weekly earnings of production and nonsupervisory employees	31	AVERAGE WEEKLY EARNINGS OF PRODUCTION AND NONSUPERVISORY EMPLOYEES
CES0500000032	5	05000000	32	S	Average hourly earnings of production and nonsupervisory employees	32	AVERAGE HOURLY EARNINGS OF PRODUCTION AND NONSUPERVISORY EMPLOYEES
CES0500000034	5	05000000	34	S	Indexes of aggregate weekly hours of production and nonsupervisory employees	34	INDEXES OF AGGREGATE WEEKLY HOURS OF PRODUCTION AND NONSUPERVISORY EMPLOYEES
CES0500000035	5	05000000	35	S	Indexes of aggregate weekly payrolls of production and nonsupervisory employees	35	INDEXES OF AGGREGATE WEEKLY PAYROLLS OF PRODUCTION AND NONSUPERVISORY EMPLOYEES
CES0500000036	5	05000000	36	S	Average weekly hours	36	AVERAGE WEEKLY HOURS
CES0500000056	5	05000000	56	S	Aggregate weekly hours of all employees	56	AGGREGATE WEEKLY HOURS OF ALL EMPLOYEES
CES0500000057	5	05000000	57	S	Aggregate weekly payrolls of all employees	57	AGGREGATE WEEKLY PAYROLLS OF ALL EMPLOYEES
CES0500000081	5	05000000	81	S	Aggregate weekly hours of production and nonsupervisory employees	81	AGGREGATE WEEKLY HOURS OF PRODUCTION AND NONSUPERVISORY EMPLOYEES
CES0500000082	5	05000000	82	S	Aggregate weekly payrolls of production and nonsupervisory employees	82	AGGREGATE WEEKLY PAYROLLS OF PRODUCTION AND NONSUPERVISORY EMPLOYEES
CES0600000001	6	06000000	1	S	All employees	1	ALL EMPLOYEES
CES0600000002	6	06000000	2	S	Average weekly hours of all employees	2	AVERAGE WEEKLY HOURS OF ALL EMPLOYEES
CES0600000003	6	06000000	3	S	Average hourly earnings of all employees	3	AVERAGE HOURLY EARNINGS OF ALL EMPLOYEES
CES0600000006	6	06000000	6	S	Production and nonsupervisory employees	6	PRODUCTION AND NONSUPERVISORY EMPLOYEES
CES0600000007	6	06000000	7	S	Average weekly hours of production and nonsupervisory employees	7	AVERAGE WEEKLY HOURS OF PRODUCTION AND NONSUPERVISORY EMPLOYEES
CES0600000008	6	06000000	8	S	Average hourly earnings of production and nonsupervisory employees	8	AVERAGE HOURLY EARNINGS OF PRODUCTION AND NONSUPERVISORY EMPLOYEES
CES0600000010	6	06000000	10	S	Women employees	10	WOMEN EMPLOYEES
CES0600000011	6	06000000	11	S	Average weekly earnings of all employees	11	AVERAGE WEEKLY EARNINGS OF ALL EMPLOYEES
CES0600000012	6	06000000	12	S	Average weekly earnings of all employees	12	AVERAGE WEEKLY EARNINGS OF ALL EMPLOYEES
CES0600000013	6	06000000	13	S	Average hourly earnings of all employees	13	AVERAGE HOURLY EARNINGS OF ALL EMPLOYEES
CES0600000016	6	06000000	16	S	Indexes of aggregate weekly hours of all employees	16	INDEXES OF AGGREGATE WEEKLY HOURS OF ALL EMPLOYEES
CES0600000017	6	06000000	17	S	Indexes of aggregate weekly payrolls of all employees	17	INDEXES OF AGGREGATE WEEKLY PAYROLLS OF ALL EMPLOYEES
CES0600000025	6	06000000	25	S	All employees	25	ALL EMPLOYEES
CES0600000030	6	06000000	30	S	Average weekly earnings of production and nonsupervisory employees	30	AVERAGE WEEKLY EARNINGS OF PRODUCTION AND NONSUPERVISORY EMPLOYEES
CES0600000031	6	06000000	31	S	Average weekly earnings of production and nonsupervisory employees	31	AVERAGE WEEKLY EARNINGS OF PRODUCTION AND NONSUPERVISORY EMPLOYEES
CES0600000032	6	06000000	32	S	Average hourly earnings of production and nonsupervisory employees	32	AVERAGE HOURLY EARNINGS OF PRODUCTION AND NONSUPERVISORY EMPLOYEES
CES0600000034	6	06000000	34	S	Indexes of aggregate weekly hours of production and nonsupervisory employees	34	INDEXES OF AGGREGATE WEEKLY HOURS OF PRODUCTION AND NONSUPERVISORY EMPLOYEES
CES0600000035	6	06000000	35	S	Indexes of aggregate weekly payrolls of production and nonsupervisory employees	35	INDEXES OF AGGREGATE WEEKLY PAYROLLS OF PRODUCTION AND NONSUPERVISORY EMPLOYEES
*/


3. Join `series` and `industry` on `industry_code`.

In [ ]:
--Displaying top 50 results

SELECT *
FROM LaborStatisticsDB.dbo.series AS s
LEFT JOIN LaborStatisticsDB.dbo.industry AS i
ON i.industry_code = s.industry_code
ORDER BY s.series_id

/*

CES0000000001	0	00000000	1	S	All employees	0	0	-	B	Total nonfarm	0	T	1
CES0000000010	0	00000000	10	S	Women employees	0	0	-	B	Total nonfarm	0	T	1
CES0000000025	0	00000000	25	S	All employees	0	0	-	B	Total nonfarm	0	T	1
CES0000000026	0	00000000	26	S	All employees	0	0	-	B	Total nonfarm	0	T	1
CES0500000001	5	05000000	1	S	All employees	1	5000000	-	A	Total private	1	T	2
CES0500000002	5	05000000	2	S	Average weekly hours of all employees	1	5000000	-	A	Total private	1	T	2
CES0500000003	5	05000000	3	S	Average hourly earnings of all employees	1	5000000	-	A	Total private	1	T	2
CES0500000006	5	05000000	6	S	Production and nonsupervisory employees	1	5000000	-	A	Total private	1	T	2
CES0500000007	5	05000000	7	S	Average weekly hours of production and nonsupervisory employees	1	5000000	-	A	Total private	1	T	2
CES0500000008	5	05000000	8	S	Average hourly earnings of production and nonsupervisory employees	1	5000000	-	A	Total private	1	T	2
CES0500000010	5	05000000	10	S	Women employees	1	5000000	-	A	Total private	1	T	2
CES0500000011	5	05000000	11	S	Average weekly earnings of all employees	1	5000000	-	A	Total private	1	T	2
CES0500000012	5	05000000	12	S	Average weekly earnings of all employees	1	5000000	-	A	Total private	1	T	2
CES0500000013	5	05000000	13	S	Average hourly earnings of all employees	1	5000000	-	A	Total private	1	T	2
CES0500000016	5	05000000	16	S	Indexes of aggregate weekly hours of all employees	1	5000000	-	A	Total private	1	T	2
CES0500000017	5	05000000	17	S	Indexes of aggregate weekly payrolls of all employees	1	5000000	-	A	Total private	1	T	2
CES0500000019	5	05000000	19	S	Average weekly hours of all employees	1	5000000	-	A	Total private	1	T	2
CES0500000021	5	05000000	21	S	Diffusion indexes	1	5000000	-	A	Total private	1	T	2
CES0500000022	5	05000000	22	S	Diffusion indexes	1	5000000	-	A	Total private	1	T	2
CES0500000023	5	05000000	23	S	Diffusion indexes	1	5000000	-	A	Total private	1	T	2
CES0500000025	5	05000000	25	S	All employees	1	5000000	-	A	Total private	1	T	2
CES0500000026	5	05000000	26	S	All employees	1	5000000	-	A	Total private	1	T	2
CES0500000030	5	05000000	30	S	Average weekly earnings of production and nonsupervisory employees	1	5000000	-	A	Total private	1	T	2
CES0500000031	5	05000000	31	S	Average weekly earnings of production and nonsupervisory employees	1	5000000	-	A	Total private	1	T	2
CES0500000032	5	05000000	32	S	Average hourly earnings of production and nonsupervisory employees	1	5000000	-	A	Total private	1	T	2
CES0500000034	5	05000000	34	S	Indexes of aggregate weekly hours of production and nonsupervisory employees	1	5000000	-	A	Total private	1	T	2
CES0500000035	5	05000000	35	S	Indexes of aggregate weekly payrolls of production and nonsupervisory employees	1	5000000	-	A	Total private	1	T	2
CES0500000036	5	05000000	36	S	Average weekly hours	1	5000000	-	A	Total private	1	T	2
CES0500000056	5	05000000	56	S	Aggregate weekly hours of all employees	1	5000000	-	A	Total private	1	T	2
CES0500000057	5	05000000	57	S	Aggregate weekly payrolls of all employees	1	5000000	-	A	Total private	1	T	2
CES0500000081	5	05000000	81	S	Aggregate weekly hours of production and nonsupervisory employees	1	5000000	-	A	Total private	1	T	2
CES0500000082	5	05000000	82	S	Aggregate weekly payrolls of production and nonsupervisory employees	1	5000000	-	A	Total private	1	T	2
CES0600000001	6	06000000	1	S	All employees	2	6000000	-	A	Goods-producing	1	T	3
CES0600000002	6	06000000	2	S	Average weekly hours of all employees	2	6000000	-	A	Goods-producing	1	T	3
CES0600000003	6	06000000	3	S	Average hourly earnings of all employees	2	6000000	-	A	Goods-producing	1	T	3
CES0600000006	6	06000000	6	S	Production and nonsupervisory employees	2	6000000	-	A	Goods-producing	1	T	3
CES0600000007	6	06000000	7	S	Average weekly hours of production and nonsupervisory employees	2	6000000	-	A	Goods-producing	1	T	3
CES0600000008	6	06000000	8	S	Average hourly earnings of production and nonsupervisory employees	2	6000000	-	A	Goods-producing	1	T	3
CES0600000010	6	06000000	10	S	Women employees	2	6000000	-	A	Goods-producing	1	T	3
CES0600000011	6	06000000	11	S	Average weekly earnings of all employees	2	6000000	-	A	Goods-producing	1	T	3
CES0600000012	6	06000000	12	S	Average weekly earnings of all employees	2	6000000	-	A	Goods-producing	1	T	3
CES0600000013	6	06000000	13	S	Average hourly earnings of all employees	2	6000000	-	A	Goods-producing	1	T	3
CES0600000016	6	06000000	16	S	Indexes of aggregate weekly hours of all employees	2	6000000	-	A	Goods-producing	1	T	3
CES0600000017	6	06000000	17	S	Indexes of aggregate weekly payrolls of all employees	2	6000000	-	A	Goods-producing	1	T	3
CES0600000025	6	06000000	25	S	All employees	2	6000000	-	A	Goods-producing	1	T	3
CES0600000030	6	06000000	30	S	Average weekly earnings of production and nonsupervisory employees	2	6000000	-	A	Goods-producing	1	T	3
CES0600000031	6	06000000	31	S	Average weekly earnings of production and nonsupervisory employees	2	6000000	-	A	Goods-producing	1	T	3
CES0600000032	6	06000000	32	S	Average hourly earnings of production and nonsupervisory employees	2	6000000	-	A	Goods-producing	1	T	3
CES0600000034	6	06000000	34	S	Indexes of aggregate weekly hours of production and nonsupervisory employees	2	6000000	-	A	Goods-producing	1	T	3
CES0600000035	6	06000000	35	S	Indexes of aggregate weekly payrolls of production and nonsupervisory employees	2	6000000	-	A	Goods-producing	1	T	3

*/


## Subqueries, Unions, Derived Tables, Oh My!

1. Write a query that returns the `series_id`, `industry_code`, `industry_name`, and `value` from the `january_2017` table but only if that value is greater than the average value for `annual_2016` of `data_type_code` 82.

In [ ]:
SELECT DISTINCT(s.series_id), i.industry_code, i.industry_name, j.value
FROM LaborStatisticsDB.dbo.january_2017 AS j
INNER JOIN LaborStatisticsDB.dbo.series AS s
ON s.series_id = j.series_id
INNER JOIN LaborStatisticsDB.dbo.industry AS i
ON i.industry_code = s.industry_code
INNER JOIN LaborStatisticsDB.dbo.annual_2016 AS a
ON a.series_id = j.series_id
WHERE j.value > a.value
AND s.data_type_code = '82'

**Optional Bonus Question:** Write the above query as a common table expression!

In [ ]:
-- Optional CTE below

2. Create a `Union` table comparing average weekly earnings of production and nonsupervisory employees between `annual_2016` and `january_2017` using the data type 30.  Round to the nearest penny.  You should have a column for the average earnings and a column for the year, and the period.

In [ ]:
-- 2016: 797.2; Jan 2017: 808.53

SELECT ROUND(AVG(value),2) AS avg_payroll, year, period
FROM LaborStatisticsDB.dbo.annual_2016
WHERE series_id IN 
        (
        SELECT series_id
        FROM LaborStatisticsDB.dbo.series
        WHERE data_type_code = '30'
        )
GROUP BY year, period

UNION

SELECT ROUND(AVG(value),2) AS avg_payroll, year, period
FROM LaborStatisticsDB.dbo.january_2017
WHERE series_id IN 
        (
        SELECT series_id
        FROM LaborStatisticsDB.dbo.series
        WHERE data_type_code = '30'
        )
GROUP BY year, period

## Summarize Your Results

With what you know now about the  Bureau of Labor Statistics (BLS) Current Employment Survey (CES) results and working with the Labor Statistics Database, answer the following questions. Note that while this is subjective, you should include relevant data to back up your opinion.

1. During which time period did production and nonsupervisory employees fare better?

January 2017 had slightly higher average weekly earnings for production/non-supervisory employees ($808.53 vs $797.20), with a negligible increase in average hours worked (36.11 hours vs 36.05 hours, three minutes)

2. In which industries did production and nonsupervisory employees fare better?

The top three industries for production and nonsupervisory employee average earnings, in January 2017, were:

Reinsurance carriers ($1847.47)
Petroleum and coal products ($1812.60)
Fossil fuel electric power generation ($1797.99)

per the following query:


SELECT DISTINCT TOP 3 i.industry_name, j.value AS average_earnings, j.year, j.period 
FROM LaborStatisticsDB.dbo.january_2017 AS j

INNER JOIN LaborStatisticsDB.dbo.series AS s
ON s.series_id = j.series_id

INNER JOIN LaborStatisticsDB.dbo.industry AS i
ON i.industry_code = s.industry_code

WHERE j.series_id IN
    (
        SELECT DISTINCT series_id
        FROM LaborStatisticsDB.dbo.series
        WHERE data_type_code = '30'
    )
AND original_file NOT LIKE '%Allces%'
ORDER BY "value" DESC ;

3. Now that you have explored the datasets, is there any data or information that you wish you had in this analysis?